In [1]:
pip install jupyterlab

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install notebook

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

df = pd.read_csv("ai4i2020.csv")
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [5]:
df["Failure"] = df["Machine failure"]

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df[[
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]]

y = df["Failure"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

df["Failure_Probability"] = model.predict_proba(X)[:,1]

In [7]:
df["Risk Score"] = df["Failure_Probability"] * 100

In [8]:
def decision_rule(score):
    if score > 70:
        return "IMMEDIATE ACTION"
    elif score > 40:
        return "SCHEDULE MAINTENANCE"
    else:
        return "NO ACTION"

df["Decision"] = df["Risk Score"].apply(decision_rule)

In [9]:
# Assume cost per failure
cost_per_failure = 5000  # USD

# Calculate expected cost
df["Expected Cost"] = df["Failure_Probability"] * cost_per_failure

# Savings logic (if action taken, prevent failure)
df["Savings"] = df.apply(
    lambda row: cost_per_failure if row["Decision"] == "IMMEDIATE ACTION" else 0,
    axis=1
)

# Total metrics
total_failures = df["Failure"].sum()
total_expected_cost = df["Expected Cost"].sum()
total_savings = df["Savings"].sum()

print("Total Failures:", total_failures)
print("Expected Cost:", round(total_expected_cost, 2))
print("Potential Savings:", total_savings)

Total Failures: 339
Expected Cost: 1721206.0
Potential Savings: 230000


In [10]:
print("\n--- EXECUTIVE INSIGHTS ---")

high_risk = df[df["Decision"] == "IMMEDIATE ACTION"].shape[0]

print(f"High-risk machines requiring immediate action: {high_risk}")
print(f"Potential cost savings: ${int(total_savings)}")
print("Recommendation: Prioritize maintenance for high-risk machines to reduce downtime.")


--- EXECUTIVE INSIGHTS ---
High-risk machines requiring immediate action: 46
Potential cost savings: $230000
Recommendation: Prioritize maintenance for high-risk machines to reduce downtime.


In [11]:
df.to_csv("executive_decision_output.csv", index=False)